# Assignment 1: Sentence Classification with Deep Learning

This notebook covers all 5 tasks of Assignment 1, progressively building from a basic RNN to an attention-based BiGRU model.

| Task | Description |
|------|-------------|
| 1 | Configuration Optimization: packed sequences, regularization, hyperparameter search |
| 2 | Input Embedding: Pre-trained GloVe word vectors |
| 3 | Output Embedding: Mean pooling sentence representation |
| 4 | Architecture Optimization: BiGRU, BiLSTM, TextCNN |
| 5 | Critical Thinking: Attention mechanism |

In [ ]:
import torch
import numpy as np
import random
import time

import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:    
    device = torch.device('cpu')
    
print(f"Using device: {device}")

In [ ]:
# Install required packages
!pip install torchtext==0.4.0 -q
!python -m spacy download en_core_web_sm -q

In [ ]:
from torchtext import data, datasets

# TEXT field: tokenize with spaCy, keep sequence lengths for packed padding
TEXT = data.Field(
    tokenize='spacy',
    tokenizer_language='en_core_web_sm',
    include_lengths=True   # returns (tensor, lengths) for packed padded sequences
)

# LABEL field for multi-class classification
LABEL = data.LabelField()

In [ ]:
# Load TREC question classification dataset (coarse-grained, 6 classes)
train_data, test_data = datasets.TREC.splits(TEXT, LABEL, fine_grained=False)

# Create 80/20 train/validation split
train_data, valid_data = train_data.split(
    split_ratio=0.8,
    random_state=random.seed(SEED)
)

print(f'Training examples  : {len(train_data):,}')
print(f'Validation examples: {len(valid_data):,}')
print(f'Test examples      : {len(test_data):,}')
print(f'\nSample: {vars(train_data.examples[0])}')

In [ ]:
MAX_VOCAB_SIZE = 10_000
BATCH_SIZE     = 64

TEXT.build_vocab(train_data, max_size=MAX_VOCAB_SIZE)
LABEL.build_vocab(train_data)

VOCAB_SIZE   = len(TEXT.vocab)
OUTPUT_DIM   = len(LABEL.vocab)
PAD_IDX      = TEXT.vocab.stoi[TEXT.pad_token]
UNK_IDX      = TEXT.vocab.stoi[TEXT.unk_token]

print(f"TEXT  vocab size : {VOCAB_SIZE}")
print(f"LABEL vocab size : {OUTPUT_DIM}")
print(f"Labels           : {dict(LABEL.vocab.stoi)}")
print(f"PAD_IDX={PAD_IDX}, UNK_IDX={UNK_IDX}")

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data),
    batch_size=BATCH_SIZE,
    sort_within_batch=True,
    device=device
)

In [ ]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_epoch(model, iterator, optimizer, criterion):
    """One full pass over the training data."""
    epoch_loss, total_correct, total_instances = 0, 0, 0
    model.train()
    for batch in iterator:
        optimizer.zero_grad()
        text, text_lengths = batch.text
        predictions = model(text, text_lengths)
        loss = criterion(predictions, batch.label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        _, pred_cls = predictions.max(dim=1)
        total_correct += (pred_cls == batch.label).sum().item()
        total_instances += batch.label.size(0)
    return epoch_loss / len(iterator), total_correct / total_instances


def evaluate_epoch(model, iterator, criterion):
    """Evaluate on a given iterator (no gradient computation)."""
    epoch_loss, total_correct, total_instances = 0, 0, 0
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            text, text_lengths = batch.text
            predictions = model(text, text_lengths)
            loss = criterion(predictions, batch.label)
            epoch_loss += loss.item()
            _, pred_cls = predictions.max(dim=1)
            total_correct += (pred_cls == batch.label).sum().item()
            total_instances += batch.label.size(0)
    return epoch_loss / len(iterator), total_correct / total_instances


def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    return int(elapsed / 60), int(elapsed % 60)


def run_training(model, train_iter, valid_iter, optimizer, criterion,
                 n_epochs, save_path=None, verbose=True,
                 patience=None):
    """
    Train for up to n_epochs, saving the best model (lowest valid loss).

    Args:
        patience (int | None):
            Early-stopping patience. Training stops if valid_loss does not
            improve for this many consecutive epochs.
            Set to None (default) to disable early stopping.

    Returns:
        best_valid_loss  (float)
        best_valid_acc   (float)
        train_losses     (list[float])  -- per-epoch training loss
        val_losses       (list[float])  -- per-epoch validation loss
        epochs_run       (int)          -- actual number of epochs completed
    """
    best_valid_loss  = float('inf')
    best_valid_acc   = 0.0
    train_losses     = []
    val_losses       = []
    patience_counter = 0

    for epoch in range(n_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_iter, optimizer, criterion)
        vl_loss, vl_acc = evaluate_epoch(model, valid_iter, criterion)
        t1 = time.time()
        mins, secs = epoch_time(t0, t1)

        train_losses.append(round(tr_loss, 6))
        val_losses.append(round(vl_loss, 6))

        improved = vl_loss < best_valid_loss
        if improved:
            best_valid_loss  = vl_loss
            best_valid_acc   = vl_acc
            patience_counter = 0
            if save_path:
                torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1

        if verbose:
            patience_info = f'  [patience {patience_counter}/{patience}]' if patience else ''
            marker        = ' ✓ saved' if improved else ''
            print(f'Epoch {epoch+1:02}/{n_epochs} | {mins}m {secs}s{marker}{patience_info}')
            print(f'  Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}%')
            print(f'  Valid Loss: {vl_loss:.4f}  Acc: {vl_acc*100:.2f}%')

        # Early stopping check
        if patience is not None and patience_counter >= patience:
            if verbose:
                print(f'\n>> Early stopping triggered after {epoch+1} epochs')
                print(f'   (no val_loss improvement for {patience} consecutive epochs)')
            break

    epochs_run = len(train_losses)
    return best_valid_loss, best_valid_acc, train_losses, val_losses, epochs_run


---
## Task 1: Configuration Optimization

### 1a — Packed Padded Sequences

**Problem with the original implementation:** The baseline RNN processes ALL tokens including `<pad>` tokens equally. This means the final hidden state `h_T` is computed after seeing padding tokens, corrupting the sentence representation. For shorter sentences in a batch, the model may return a hidden state dominated by padding rather than the actual last word.

**Solution:** Use `pack_padded_sequence` + `pad_packed_sequence`. PyTorch skips padding tokens during the RNN computation, so each sentence's hidden state is extracted at its actual last real token. This:
- Gives more accurate sentence representations
- Is also faster (less computation on padding)

### 1b — Regularization

Two complementary techniques are added:
1. **Dropout** — applied after the embedding and before the final linear layer. Randomly zeroes activations during training, forcing the model to learn redundant representations and reducing co-adaptation.
2. **Weight Decay (L2 regularization)** — added to the optimizer. Penalises large weights, keeps the model from overfitting.
3. **Gradient Clipping** — already included in `run_training`; prevents the exploding gradient problem common in RNNs.

In [ ]:
class RNN_Task1(nn.Module):
    """
    Vanilla RNN with:
      - packed padded sequences  (Task 1a)
      - dropout regularisation   (Task 1b)
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 dropout=0.2, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn       = nn.RNN(embedding_dim, hidden_dim, batch_first=False)
        self.fc        = nn.Linear(hidden_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))

        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(),
                                          enforce_sorted=False)
        packed_out, hidden = self.rnn(packed_emb)

        hidden = self.dropout(hidden.squeeze(0))
        # hidden        = [batch, hidden_dim]
        return self.fc(hidden)


print("RNN_Task1 model defined.")

### Task 1c — Hyperparameter Search

We search over: optimizer type, learning rate, hidden dimension, dropout rate, and weight decay.
Each configuration is trained for **5 epochs** on the validation set to efficiently compare. The best configuration is then trained for the full **10 epochs**.

In [ ]:
import itertools, os, json

DATA_DIR     = os.path.join('.data')
os.makedirs(DATA_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(DATA_DIR, 'Task1_HyperparameterTuning.json')

# Load existing results so a crashed run can be resumed without re-running configs
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as _f:
        all_results = json.load(_f)
    print(f'Loaded {len(all_results)} existing result(s) from {RESULTS_FILE}')
else:
    all_results = []
    print(f'Starting fresh — results will be saved to: {RESULTS_FILE}')

already_run = {r['config']['name'] for r in all_results}

# ── Search grid ────────────────────────────────────────────────────────────────
SEARCH_EPOCHS   = 20
EARLY_STOP_PAT  = 3          # stop a config if no val improvement for 5 epochs

opt_configs     = ['SGD', 'Adam']
lr_configs      = [0.01, 1e-3, 1e-4]
hidden_configs  = [50, 100, 256, 512]
dropout_configs = [0.0, 0.2, 0.3, 0.4]
wd_configs      = [0, 1e-4]

criterion    = nn.CrossEntropyLoss().to(device)
search_results = []   # results from THIS run (not including resumed ones)

total = (len(opt_configs) * len(lr_configs) *
         len(hidden_configs) * len(dropout_configs) * len(wd_configs))
done  = 0

for (opt_type, lr, hid_dim, dropout, wd) in itertools.product(
        opt_configs, lr_configs, hidden_configs, dropout_configs, wd_configs):

    done       += 1
    config_name = f'{opt_type}_lr={lr}_hid={hid_dim}_drop={dropout}_wd={wd}'

    # Skip configs already persisted (resume support)
    if config_name in already_run:
        print(f'[{done:>4}/{total}] SKIP (already saved): {config_name}')
        continue

    print(f'\n[{done:>4}/{total}] {config_name}')

    torch.manual_seed(SEED)
    m = RNN_Task1(VOCAB_SIZE, 100, hid_dim, OUTPUT_DIM,
                  dropout=dropout, pad_idx=PAD_IDX).to(device)

    if opt_type == 'Adam':
        opt = optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    else:
        opt = optim.SGD(m.parameters(), lr=lr, weight_decay=wd)

    # Train — early stopping enabled
    best_vl, best_vacc, tr_losses, vl_losses, epochs_run = run_training(
        m, train_iterator, valid_iterator, opt, criterion,
        n_epochs   = SEARCH_EPOCHS,
        save_path  = None,
        verbose    = False,
        patience   = EARLY_STOP_PAT
    )

    stopped_early = epochs_run < SEARCH_EPOCHS
    tag = '  (early stop)' if stopped_early else ''
    print(f'   epochs={epochs_run:>2} | val_acc={best_vacc*100:.2f}%'
          f' | val_loss={best_vl:.4f}{tag}')

    # ── Build result record ─────────────────────────────────────────────────────
    result = {
        'config': {
            'name'      : config_name,
            'opt_type'  : opt_type,
            'lr'        : lr,
            'hidden_dim': hid_dim,
            'dropout'   : dropout,
            'wd'        : wd,
        },
        'num_epochs'     : epochs_run,
        'stopped_early'  : stopped_early,
        'best_valid_loss': round(best_vl,    6),
        'best_valid_acc' : round(best_vacc,  6),
        'train_losses'   : tr_losses,
        'val_losses'     : vl_losses,
    }

    all_results.append(result)
    search_results.append(result)

    # Persist after every config — safe against mid-run crashes
    with open(RESULTS_FILE, 'w') as _f:
        json.dump(all_results, _f, indent=2)

print(f'\nAll done. {len(search_results)} new config(s) evaluated.')
print(f'Results saved to: {RESULTS_FILE}')

# ── Pick best across ALL results (including any resumed ones) ───────────────────
best_row  = max(all_results, key=lambda r: r['best_valid_acc'])
BEST_NAME = best_row['config']['name']
BEST_OPT  = best_row['config']['opt_type']
BEST_LR   = best_row['config']['lr']
BEST_EMB  = 100
BEST_HID  = best_row['config']['hidden_dim']
BEST_DROP = best_row['config']['dropout']
BEST_WD   = best_row['config']['wd']

print(f"\n{'='*60}")
print(f"Best config  : {BEST_NAME}")
print(f"Valid Acc    : {best_row['best_valid_acc']*100:.2f}%")
print(f"Epochs run   : {best_row['num_epochs']}")
print(f"Results file : {RESULTS_FILE}")
print(f"{'='*60}")


In [ ]:
N_EPOCHS = 100

torch.manual_seed(SEED)
model_t1 = RNN_Task1(VOCAB_SIZE, BEST_EMB, BEST_HID, OUTPUT_DIM,
                     dropout=BEST_DROP, pad_idx=PAD_IDX).to(device)

if BEST_OPT == 'Adam':
    opt_t1 = optim.Adam(model_t1.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
else:
    opt_t1 = optim.SGD(model_t1.parameters(), lr=BEST_LR, weight_decay=BEST_WD)

print(f"Task 1 model — {count_parameters(model_t1):,} trainable parameters")
print(f"Best config: {BEST_NAME}\n")

run_training(model_t1, train_iterator, valid_iterator, opt_t1, criterion,
             N_EPOCHS, save_path='models/task1-model.pt', patience= EARLY_STOP_PAT)

In [ ]:
model_t1.load_state_dict(torch.load('models/task1-model.pt'))

_, valid_acc_t1 = evaluate_epoch(model_t1, valid_iterator, criterion)
_, test_acc_t1  = evaluate_epoch(model_t1, test_iterator,  criterion)

print("=" * 50)
print("Task 1 Results (RNN + Packed Seq + Dropout + Best Hyper)")
print(f"  Valid Acc : {valid_acc_t1 *100:.2f}%")
print(f"  Test  Acc : {test_acc_t1 *100:.2f}%")
print("=" * 50)

---
## Task 2: Pre-trained Input Embeddings (GloVe)

We replace randomly initialised word embeddings with **GloVe 6B 100-dimensional** vectors.

**Why GloVe?** GloVe (Global Vectors for Word Representation) was trained on 6 billion tokens from Wikipedia + Gigaword. Its vectors encode rich semantic and syntactic relationships that random initialisation cannot provide out-of-the-box. For small datasets like TREC (~4,000 training sentences), transferring this external knowledge is especially beneficial because the model cannot learn good representations from scratch.

The `<unk>` and `<pad>` token embeddings are zeroed out after initialisation to avoid spurious signals.

In [ ]:
# Rebuild vocabulary with GloVe 100-d vectors
# (same words, same indices — just adds pretrained vectors)
print("Loading GloVe 6B 100d vectors (download ~330 MB on first run)...")
try:
    TEXT.build_vocab(
        train_data,
        max_size=MAX_VOCAB_SIZE,
        vectors="glove.6B.100d",
        unk_init=torch.Tensor.normal_  # random normal for OOV words
    )
    EMBEDDING_DIM = 100
    print(f"GloVe loaded. Vector matrix: {TEXT.vocab.vectors.shape}")
except Exception as e:
    print(f"GloVe download failed ({e}). Falling back to random init.")
    TEXT.build_vocab(train_data, max_size=MAX_VOCAB_SIZE)
    EMBEDDING_DIM = 100

VOCAB_SIZE = len(TEXT.vocab)
PAD_IDX    = TEXT.vocab.stoi[TEXT.pad_token]
UNK_IDX    = TEXT.vocab.stoi[TEXT.unk_token]

print(f"Vocab size: {VOCAB_SIZE}")

In [ ]:
torch.manual_seed(SEED)
model_t2 = RNN_Task1(VOCAB_SIZE, EMBEDDING_DIM, BEST_HID, OUTPUT_DIM,
                     dropout=BEST_DROP, pad_idx=PAD_IDX).to(device)

# Initialise with pretrained vectors (if available)
if hasattr(TEXT.vocab, 'vectors') and TEXT.vocab.vectors is not None:
    model_t2.embedding.weight.data.copy_(TEXT.vocab.vectors)
    # Zero out special tokens so they don't contribute noise
    model_t2.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    model_t2.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)
    print("Pretrained embeddings initialised.")
else:
    print("Using random initialisation (GloVe not available).")

if BEST_OPT == 'Adam':
    opt_t2 = optim.Adam(model_t2.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
else:
    opt_t2 = optim.SGD(model_t2.parameters(), lr=BEST_LR, weight_decay=BEST_WD)

print(f"Task 2 model — {count_parameters(model_t2):,} trainable parameters\n")

run_training(model_t2, train_iterator, valid_iterator, opt_t2, criterion,
             N_EPOCHS, save_path='models/task2-model.pt', patience=5)

In [ ]:
model_t2.load_state_dict(torch.load('task2-model.pt'))

_, valid_acc_t2 = evaluate_epoch(model_t2, valid_iterator, criterion)
_, test_acc_t2  = evaluate_epoch(model_t2, test_iterator,  criterion)

print("=" * 60)
print("Task 2 Results (+ GloVe pre-trained embeddings)")
print(f"  Valid Acc : {valid_acc_t2*100:.2f}%  (Task 1: {valid_acc_t1*100:.2f}%)")
print(f"  Test  Acc : {test_acc_t2 *100:.2f}%  (Task 1: {test_acc_t1 *100:.2f}%)")
print(f"  Delta Valid : {(valid_acc_t2-valid_acc_t1)*100:+.2f}%")
print(f"  Delta Test  : {(test_acc_t2 -test_acc_t1 )*100:+.2f}%")
print("=" * 60)

---
## Task 3: Output Embedding — Mean Pooling

The baseline uses only the **final** hidden state `h_T` as the sentence representation. This forces the model to compress the entire sentence into one step, which can lose information for longer or complex sentences.

**Alternative — Mean Pooling:** Average all non-padded hidden states `{h_1, …, h_T}`. This gives each token an equal vote in the sentence representation and can better capture sentence-level semantics than relying solely on the last token.

Other valid options (not implemented here but noted):
- **Max Pooling:** take the maximum activation across time steps — highlights the most salient features.
- **Attention-weighted sum:** learnable weights over positions (implemented in Task 5).

In [ ]:
class RNN_MeanPool(nn.Module):
    """
    RNN with mean pooling over non-padded hidden states (Task 3).
    Otherwise identical architecture to RNN_Task1.
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn       = nn.RNN(embedding_dim, hidden_dim, batch_first=False)
        self.fc        = nn.Linear(hidden_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        # embedded = [sent_len, batch, emb_dim]

        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(),
                                          enforce_sorted=False)
        packed_out, _ = self.rnn(packed_emb)
        output, out_lengths = pad_packed_sequence(packed_out)
        # output   = [sent_len, batch, hidden_dim]

        # Permute to [batch, sent_len, hidden_dim]
        output = output.permute(1, 0, 2)

        # Build mask: True for real (non-padded) positions
        lengths_dev = out_lengths.to(device).float()          # [batch]
        max_len     = output.size(1)
        mask = (torch.arange(max_len, device=device)
                     .unsqueeze(0) < lengths_dev.long().unsqueeze(1))
        # mask = [batch, sent_len]

        # Zero-out padding then average
        masked_out  = output * mask.unsqueeze(2).float()      # [batch, sent_len, hidden_dim]
        mean_pooled = masked_out.sum(dim=1) / lengths_dev.unsqueeze(1)
        # mean_pooled = [batch, hidden_dim]

        mean_pooled = self.dropout(mean_pooled)
        return self.fc(mean_pooled)


print("RNN_MeanPool model defined.")

In [ ]:
torch.manual_seed(SEED)
model_t3 = RNN_MeanPool(VOCAB_SIZE, EMBEDDING_DIM, BEST_HID, OUTPUT_DIM,
                         dropout=BEST_DROP, pad_idx=PAD_IDX).to(device)

# Initialise with same GloVe vectors
if hasattr(TEXT.vocab, 'vectors') and TEXT.vocab.vectors is not None:
    model_t3.embedding.weight.data.copy_(TEXT.vocab.vectors)
    model_t3.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    model_t3.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)

if BEST_OPT == 'adam':
    opt_t3 = optim.Adam(model_t3.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
else:
    opt_t3 = optim.SGD(model_t3.parameters(), lr=BEST_LR, weight_decay=BEST_WD)

print(f"Task 3 model — {count_parameters(model_t3):,} trainable parameters\n")

run_training(model_t3, train_iterator, valid_iterator, opt_t3, criterion,
             N_EPOCHS, save_path='task3-model.pt')

In [ ]:
model_t3.load_state_dict(torch.load('task3-model.pt'))

_, valid_acc_t3 = evaluate_epoch(model_t3, valid_iterator, criterion)
_, test_acc_t3  = evaluate_epoch(model_t3, test_iterator,  criterion)

print("=" * 60)
print("Task 3 Results (+ Mean Pooling sentence embedding)")
print(f"  Valid Acc : {valid_acc_t3*100:.2f}%  (Task 2: {valid_acc_t2*100:.2f}%)")
print(f"  Test  Acc : {test_acc_t3 *100:.2f}%  (Task 2: {test_acc_t2 *100:.2f}%)")
print(f"  Delta Valid : {(valid_acc_t3-valid_acc_t2)*100:+.2f}%")
print(f"  Delta Test  : {(test_acc_t3 -test_acc_t2 )*100:+.2f}%")
print("=" * 60)
print()
print("Discussion: Mean pooling considers all hidden states equally,")
print("which can be more informative than the final hidden state —")
print("especially when key classification words appear in the middle")
print("of a question (e.g. 'What *color* is...'). However, unidirectional")
print("RNN's early hidden states lack right-context, limiting the benefit.")

---
## Task 4: Architecture Optimization

We replace the unidirectional single-layer RNN with three advanced architectures, all trained with GloVe embeddings and mean pooling for a fair comparison.

### Common settings for all Task 4 models:
- GloVe 100-d pretrained embeddings (fine-tuned)
- Hidden dim: 256, 2 layers, dropout: 0.5
- Adam optimizer, lr=1e-3, weight_decay=1e-4
- 10 epochs, gradient clipping

### 4a — Bidirectional GRU (2 hidden layers)

A **GRU** (Gated Recurrent Unit) replaces the vanilla RNN with gating mechanisms (reset gate, update gate) that help retain long-range dependencies and alleviate the vanishing gradient problem.

**Bidirectionality** processes each sentence both left-to-right and right-to-left, giving every token access to its full sentence context. The final representation concatenates both directions: `[h_forward ; h_backward]`.

In [ ]:
class BiGRU(nn.Module):
    """Bidirectional 2-layer GRU (Task 4a)."""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 n_layers=2, dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(
            embedding_dim, hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=False
        )
        self.fc      = nn.Linear(hidden_dim * 2, output_dim)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(),
                                          enforce_sorted=False)
        packed_out, hidden = self.rnn(packed_emb)
        # hidden = [n_layers * 2, batch, hidden_dim]

        # Take the last layer's forward (hidden[-2]) and backward (hidden[-1]) states
        hidden = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        # hidden = [batch, hidden_dim * 2]
        hidden = self.dropout(hidden)
        return self.fc(hidden)


ARCH_HIDDEN = 256
ARCH_DROP   = 0.5
ARCH_LR     = 1e-3
ARCH_WD     = 1e-4

torch.manual_seed(SEED)
model_bigru = BiGRU(VOCAB_SIZE, EMBEDDING_DIM, ARCH_HIDDEN, OUTPUT_DIM,
                    n_layers=2, dropout=ARCH_DROP, pad_idx=PAD_IDX).to(device)

if hasattr(TEXT.vocab, 'vectors') and TEXT.vocab.vectors is not None:
    model_bigru.embedding.weight.data.copy_(TEXT.vocab.vectors)
    model_bigru.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    model_bigru.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)

opt_bigru = optim.Adam(model_bigru.parameters(), lr=ARCH_LR, weight_decay=ARCH_WD)
print(f"BiGRU model — {count_parameters(model_bigru):,} trainable parameters\n")

run_training(model_bigru, train_iterator, valid_iterator, opt_bigru, criterion,
             N_EPOCHS, save_path='bigru-model.pt')

In [ ]:
model_bigru.load_state_dict(torch.load('bigru-model.pt'))
_, valid_acc_bigru = evaluate_epoch(model_bigru, valid_iterator, criterion)
_, test_acc_bigru  = evaluate_epoch(model_bigru, test_iterator,  criterion)
print(f"BiGRU  — Valid: {valid_acc_bigru*100:.2f}%  Test: {test_acc_bigru*100:.2f}%")

### 4b — Bidirectional LSTM (2 hidden layers)

The **LSTM** (Long Short-Term Memory) adds an explicit cell state `c_t` alongside the hidden state `h_t`. The cell state acts as a long-term memory that can be selectively read, written, or erased through input, forget, and output gates. This makes LSTMs particularly strong at capturing dependencies over long sequences.

In [ ]:
class BiLSTM(nn.Module):
    """Bidirectional 2-layer LSTM (Task 4b)."""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 n_layers=2, dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=False
        )
        self.fc      = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(),
                                          enforce_sorted=False)
        packed_out, (hidden, cell) = self.rnn(packed_emb)
        # hidden = [n_layers * 2, batch, hidden_dim]

        hidden = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        hidden = self.dropout(hidden)
        return self.fc(hidden)


torch.manual_seed(SEED)
model_bilstm = BiLSTM(VOCAB_SIZE, EMBEDDING_DIM, ARCH_HIDDEN, OUTPUT_DIM,
                       n_layers=2, dropout=ARCH_DROP, pad_idx=PAD_IDX).to(device)

if hasattr(TEXT.vocab, 'vectors') and TEXT.vocab.vectors is not None:
    model_bilstm.embedding.weight.data.copy_(TEXT.vocab.vectors)
    model_bilstm.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    model_bilstm.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)

opt_bilstm = optim.Adam(model_bilstm.parameters(), lr=ARCH_LR, weight_decay=ARCH_WD)
print(f"BiLSTM model — {count_parameters(model_bilstm):,} trainable parameters\n")

run_training(model_bilstm, train_iterator, valid_iterator, opt_bilstm, criterion,
             N_EPOCHS, save_path='bilstm-model.pt')

In [ ]:
model_bilstm.load_state_dict(torch.load('bilstm-model.pt'))
_, valid_acc_bilstm = evaluate_epoch(model_bilstm, valid_iterator, criterion)
_, test_acc_bilstm  = evaluate_epoch(model_bilstm, test_iterator,  criterion)
print(f"BiLSTM — Valid: {valid_acc_bilstm*100:.2f}%  Test: {test_acc_bilstm*100:.2f}%")

### 4c — TextCNN

A **Convolutional Neural Network** treats the sentence as an "image" of shape `[batch, 1, seq_len, emb_dim]`. Multiple 1-D convolutions with different **filter sizes** act as n-gram detectors — a filter of width 3 slides over every 3-gram in the sentence. Max-over-time pooling then selects the most prominent feature per filter.

**Configuration:** filter sizes `[2, 3, 4]`, 100 filters each → 300 features total → FC → 6 classes.

CNNs are **non-sequential**: they see all n-grams in parallel, making them fast and effective for short-text classification tasks like TREC.

In [ ]:
class TextCNN(nn.Module):
    """
    Kim (2014)-style TextCNN (Task 4c).
    Multiple filter widths → max pooling → FC.
    """
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes,
                 output_dim, dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        # Conv2d: (batch, 1, seq_len, emb_dim) -> (batch, n_filters, seq_len-fs+1, 1)
        self.convs = nn.ModuleList([
            nn.Conv2d(in_channels=1, out_channels=n_filters,
                      kernel_size=(fs, embedding_dim))
            for fs in filter_sizes
        ])
        self.fc      = nn.Linear(len(filter_sizes) * n_filters, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        # text = [sent_len, batch]  →  [batch, sent_len]
        text     = text.permute(1, 0)
        embedded = self.dropout(self.embedding(text))
        # embedded = [batch, sent_len, emb_dim]

        embedded = embedded.unsqueeze(1)
        # embedded = [batch, 1, sent_len, emb_dim]

        # Apply each conv + relu, squeeze last dim
        conved = [torch.relu(conv(embedded)).squeeze(3) for conv in self.convs]
        # conved[i] = [batch, n_filters, sent_len - fs + 1]

        # Max-over-time pooling
        pooled = [torch.max_pool1d(c, c.shape[2]).squeeze(2) for c in conved]
        # pooled[i] = [batch, n_filters]

        cat = self.dropout(torch.cat(pooled, dim=1))
        # cat = [batch, n_filters * len(filter_sizes)]
        return self.fc(cat)


N_FILTERS    = 100
FILTER_SIZES = [2, 3, 4]

torch.manual_seed(SEED)
model_cnn = TextCNN(VOCAB_SIZE, EMBEDDING_DIM, N_FILTERS, FILTER_SIZES,
                    OUTPUT_DIM, dropout=ARCH_DROP, pad_idx=PAD_IDX).to(device)

if hasattr(TEXT.vocab, 'vectors') and TEXT.vocab.vectors is not None:
    model_cnn.embedding.weight.data.copy_(TEXT.vocab.vectors)
    model_cnn.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    model_cnn.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)

opt_cnn = optim.Adam(model_cnn.parameters(), lr=ARCH_LR, weight_decay=ARCH_WD)
print(f"TextCNN model — {count_parameters(model_cnn):,} trainable parameters\n")

run_training(model_cnn, train_iterator, valid_iterator, opt_cnn, criterion,
             N_EPOCHS, save_path='cnn-model.pt')

In [ ]:
model_cnn.load_state_dict(torch.load('cnn-model.pt'))
_, valid_acc_cnn = evaluate_epoch(model_cnn, valid_iterator, criterion)
_, test_acc_cnn  = evaluate_epoch(model_cnn, test_iterator,  criterion)
print(f"CNN    — Valid: {valid_acc_cnn*100:.2f}%  Test: {test_acc_cnn*100:.2f}%")

In [ ]:
print("=" * 65)
print("Task 4 Architecture Comparison (all with GloVe, Task 3 baseline)")
print("-" * 65)
print(f"{'Model':<30} {'Valid Acc':>10} {'Test Acc':>10}")
print("-" * 65)
print(f"{'Task 3  (RNN MeanPool)     ':<30} {valid_acc_t3*100:>9.2f}% {test_acc_t3*100:>9.2f}%")
print(f"{'Task 4a (BiGRU  2-layer)   ':<30} {valid_acc_bigru*100:>9.2f}% {test_acc_bigru*100:>9.2f}%")
print(f"{'Task 4b (BiLSTM 2-layer)   ':<30} {valid_acc_bilstm*100:>9.2f}% {test_acc_bilstm*100:>9.2f}%")
print(f"{'Task 4c (TextCNN [2,3,4])  ':<30} {valid_acc_cnn*100:>9.2f}% {test_acc_cnn*100:>9.2f}%")
print("=" * 65)
print()
print("Discussion:")
print("BiGRU and BiLSTM benefit from bidirectionality and gating; they")
print("outperform the basic RNN because each token sees full sentence")
print("context. LSTM's cell state gives it a slight edge over GRU on")
print("complex question structures.")
print("TextCNN excels because TREC questions are short and local n-gram")
print("patterns (e.g. 'Who is', 'How many') are highly discriminative.")
print("Parallel computation makes CNN faster than sequential models.")

---
## Task 5: Critical Thinking — Additive Self-Attention over BiGRU

**Technique:** We add an **additive attention mechanism** (Bahdanau-style) on top of the BiGRU from Task 4a.

**Why not just use the last hidden state?** Not every question has its most discriminative word at the end. Attention learns a *soft weighting* over all hidden states:

$$\alpha_t = \text{softmax}\left(\mathbf{w}^\top \mathbf{h}_t\right)$$

$$\mathbf{s} = \sum_t \alpha_t \mathbf{h}_t$$

The context vector `s` focuses on the most informative words for classification. This is **non-trivial** because the weights are learned, not hand-crafted, and the mechanism is differentiable end-to-end.

**This is different from Transformers:** We add attention only on the *output* of the BiGRU, without multi-head self-attention layers or positional encodings. The BiGRU still provides the sequential inductive bias.

In [ ]:
class AttentionBiGRU(nn.Module):
    """
    BiGRU (2-layer) + additive attention over all hidden states (Task 5).
    Instead of using only the final hidden state, attention learns a
    soft weighting over the full output sequence.
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 n_layers=2, dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(
            embedding_dim, hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=False
        )
        # Attention: project each hidden state to a scalar score
        self.attention = nn.Linear(hidden_dim * 2, 1, bias=False)
        self.fc         = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        # embedded = [sent_len, batch, emb_dim]

        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(),
                                          enforce_sorted=False)
        packed_out, _ = self.rnn(packed_emb)
        output, out_lengths = pad_packed_sequence(packed_out)
        # output = [sent_len, batch, hidden_dim*2]

        output = output.permute(1, 0, 2)
        # output = [batch, sent_len, hidden_dim*2]

        # --- Attention ---
        scores = self.attention(output).squeeze(2)
        # scores = [batch, sent_len]

        # Mask padding positions with -inf before softmax
        lengths_dev = out_lengths.to(device)
        mask = (torch.arange(output.size(1), device=device)
                     .unsqueeze(0) >= lengths_dev.unsqueeze(1))
        scores = scores.masked_fill(mask, float('-inf'))

        weights = torch.softmax(scores, dim=1).unsqueeze(2)
        # weights = [batch, sent_len, 1]

        # Weighted sum (context vector)
        context = (output * weights).sum(dim=1)
        # context = [batch, hidden_dim*2]

        context = self.dropout(context)
        return self.fc(context)


torch.manual_seed(SEED)
model_attn = AttentionBiGRU(VOCAB_SIZE, EMBEDDING_DIM, ARCH_HIDDEN, OUTPUT_DIM,
                              n_layers=2, dropout=ARCH_DROP, pad_idx=PAD_IDX).to(device)

if hasattr(TEXT.vocab, 'vectors') and TEXT.vocab.vectors is not None:
    model_attn.embedding.weight.data.copy_(TEXT.vocab.vectors)
    model_attn.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    model_attn.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)

opt_attn = optim.Adam(model_attn.parameters(), lr=ARCH_LR, weight_decay=ARCH_WD)
print(f"AttentionBiGRU — {count_parameters(model_attn):,} trainable parameters\n")

run_training(model_attn, train_iterator, valid_iterator, opt_attn, criterion,
             N_EPOCHS, save_path='attn-model.pt')

In [ ]:
model_attn.load_state_dict(torch.load('attn-model.pt'))
_, valid_acc_attn = evaluate_epoch(model_attn, valid_iterator, criterion)
_, test_acc_attn  = evaluate_epoch(model_attn, test_iterator,  criterion)

print(f"Attention BiGRU — Valid: {valid_acc_attn*100:.2f}%  Test: {test_acc_attn*100:.2f}%")
print()
print("Discussion: Attention allows the model to selectively focus on")
print("key words like 'who', 'where', 'how many'. Unlike mean pooling,")
print("the weights are input-dependent and learned. This is especially")
print("helpful for questions where a single word determines the class")
print("(e.g. 'WHO won...' → HUM, 'HOW MANY...' → NUM).")

---
## Final Summary of All Results

In [ ]:
print("=" * 70)
print("COMPLETE RESULTS SUMMARY — CE7455 Assignment 1")
print("=" * 70)
print(f"{'Task':<45} {'Valid Acc':>10} {'Test Acc':>10}")
print("-" * 70)
print(f"{'T1: RNN+Packed+Dropout (best hyper)':<45} "
      f"{valid_acc_t1*100:>9.2f}% {test_acc_t1*100:>9.2f}%")
print(f"{'T2: + GloVe pre-trained embeddings':<45} "
      f"{valid_acc_t2*100:>9.2f}% {test_acc_t2*100:>9.2f}%")
print(f"{'T3: + Mean pooling output embedding':<45} "
      f"{valid_acc_t3*100:>9.2f}% {test_acc_t3*100:>9.2f}%")
print(f"{'T4a: BiGRU 2-layer (last hidden)':<45} "
      f"{valid_acc_bigru*100:>9.2f}% {test_acc_bigru*100:>9.2f}%")
print(f"{'T4b: BiLSTM 2-layer (last hidden)':<45} "
      f"{valid_acc_bilstm*100:>9.2f}% {test_acc_bilstm*100:>9.2f}%")
print(f"{'T4c: TextCNN [2,3,4] x100 filters':<45} "
      f"{valid_acc_cnn*100:>9.2f}% {test_acc_cnn*100:>9.2f}%")
print(f"{'T5:  Attention BiGRU 2-layer':<45} "
      f"{valid_acc_attn*100:>9.2f}% {test_acc_attn*100:>9.2f}%")
print("=" * 70)